In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("PREP_DEEP.csv")

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
print(f"{(missing > 0.40).sum()} columns have >40% missing")

In [ ]:
missing = df.isnull().mean()
drop_cols = missing[missing > 0.40].index
df = df.drop(columns=drop_cols)

pd.Series(drop_cols).to_csv("dropped_vars_over_40pct.csv", index=False)

In [ ]:
#columns with non-zero variance
nzv_cols = df.columns[df.apply(lambda x: x.value_counts(normalize=True).max() > 0.95)]

df.drop(columns=nzv_cols, inplace=True)

pd.Series(nzv_cols).to_csv("dropped_vars_over_95pct_same_val.csv", index=False)

In [ ]:
df.shape

In [ ]:
redundant_cols = ['THORACIC_DGN', 'ABO', 'ETHNICITY', 'WGT_KG_TCR', 'HGT_CM_TCR', 'BMI_TCR', 'EDUCATION',
    'INOTROPES_TCR', 'VAD_DEVICE_TY_TCR', 'FUNC_STAT_TCR', 'PRI_PAYMENT_TCR', 'TCR_DGN',
    'MOST_RCNT_CREAT', 'INIT_STAT', 'DAYSWAIT_CHRON', 'INIT_AGE', 'LIFE_SUP_TCR',
    'INIT_HGT_CM_CALC', 'INIT_WGT_KG_CALC', 'INIT_BMI_CALC', 'END_HGT_CM_CALC',
    'END_WGT_KG_CALC', 'END_BMI_CALC', 'VENTILATOR_TCR', 'ACADEMIC_PRG_TCR',
    'ACADEMIC_LEVEL_TCR', 'PRIOR_TH_SURG_TCR', 'BW4', 'BW6', 'C1', 'C2', 'DR51', 'DR51_2',
    'DR52', 'DR52_2', 'DR53', 'DR53_2', 'DQ1', 'DQ2', 'DA1', 'DA2', 'DB1', 'DB2', 'DDR1',
    'DDR2', 'RA1', 'RA2', 'RB1', 'RB2', 'RDR1', 'RDR2', 'AMIS', 'BMIS', 'DRMIS', 'ABO_DON',
    'BMI_DON_CALC', 'BMI_CALC', 
    'VENT_SUPPORT_AFTER_LIST', 'TX_YEAR', 'OPO_CTR_CODE',
    'INIT_OPO_CTR_CODE', 'END_OPO_CTR_CODE', 'LISTING_CTR_CODE'
   ]

unnecessary_cols = ['PERM_STATE', 'ACTIVATE_DATE', 'END_DATE', 'INIT_DATE', 'PT_CODE', 'REGION', 'WL_ID_CODE',
    'VAL_DT_TCR', 'ADMISSION_DATE', 'PERM_STATE_TRR', 
    'INOTROP_VASO_MN_TRR',
    'DON_RETYP', 'CITIZENSHIP_DON', 'HOME_STATE_DON',
    'DEATH_CIRCUM_DON', 'DEATH_MECH_DON', 'RECOVERY_DATE_DON', 'SHARE_TY', 'DISTANCE',
    'TRR_ID_CODE', 'VAL_DT_TRR', 'ADMIT_DATE_DON', 
    'TATTOOS', 
    'VAL_DT_DDR', 'DONOR_ID',
    'REFERRAL_DATE', 'LISTYR', 'CHEST_XRAY_DON']

outcome_cols = ["ACUTE_REJ_EPI", "PST_DIAL", "LASTFUNO", "PSTATUS", "PTIME", "PX_STAT", "FUNC_STAT_TRF",
    "TRTREJ1Y", "PX_STAT_DATE", "DISCHARGE_DATE", 
    "GRF_STAT", 
    "LOS"]



In [ ]:
cols_to_drop = redundant_cols + unnecessary_cols + outcome_cols
df = df.drop(columns=cols_to_drop)
df.shape

In [ ]:
df = df.apply(lambda col: col.map(lambda x: 1 if isinstance(x, str) and x.strip() == "P" else x))

df = df.apply(lambda col: col.map(lambda x: 1 if isinstance(x, str) and x.strip() == "Y" else x))

df = df.apply(lambda col: col.map(lambda x: 1 if isinstance(x, str) and x.strip() == "F" else x))

df = df.apply(lambda col: col.map(lambda x: 0 if isinstance(x, str) and x.strip() == "N" else x))

df = df.apply(lambda col: col.map(lambda x: 0 if isinstance(x, str) and x.strip() == "M" else x))

In [ ]:
spec_nan_cols = ["ETHCAT", "ETHCAT_DON", "ACADEMIC_LEVEL_TRR", "ACADEMIC_PRG_TRR", 
                "FUNC_STAT_TRR", "COGNITIVE_DEV_TRR", "MOTOR_DEV_TRR", "COD_CAD_DON"]
bad_codes = [998, 999]

df[spec_nan_cols] = df[spec_nan_cols].replace(bad_codes, np.nan)


In [ ]:
def keep_top_n_categories(s: pd.Series, n: int, other_label: str = "Other") -> pd.Series:
    """
    Keep the top-n most frequent categories in a single column; map all others to `other_label`.
    Preserves NaN as NaN.
    """
    top = s.value_counts(dropna=True).nlargest(n).index
    return s.where(s.isna() | s.isin(top), other_label)

df["DIAG"] = keep_top_n_categories(df["DIAG"], n=5, other_label=9999)
df["ETHCAT"] = keep_top_n_categories(df["ETHCAT"], n=3, other_label=9999)
df["ETHCAT_DON"] = keep_top_n_categories(df["ETHCAT_DON"], n=3, other_label=9999)

In [ ]:
private_ins = [1]
public_ins = [2, 3, 4, 5, 6, 7, 13]
other = [8, 9, 10, 11, 12, 14, 15]

In [ ]:
#divide into private insurance, public insurance, and other payment
df.loc[df['PRI_PAYMENT_TRR'] == 1, 'PRI_PAYMENT_TRR'] = 1
df.loc[df['PRI_PAYMENT_TRR'].isin(public_ins), 'PRI_PAYMENT_TRR'] = 2
df.loc[df['PRI_PAYMENT_TRR'].isin(other), 'PRI_PAYMENT_TRR'] = 9999

In [ ]:
df.to_csv("CLEAN_DEEP.csv", index=False)